# Zenith-averaged neutrino flux

Down-going muon and electron neutrinos averaged over 11 bins in
$\cos\theta$, split into conventional and prompt components. The plainest
full example: one solve loop, two plots, one ASCII file.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from MCEq import config
from MCEq.core import MCEqRun
from MCEq.notebook import apply_kernel_and_grid, download_jupyter

## 01 · Configuration

In [ ]:
# MCEq example configuration (materialized from the form)
database = 'mceq_db_v140reduced_compact.h5'
interaction_model = 'SIBYLL21'
import crflux.models as pm
primary_model = (pm.HillasGaisser2012, 'H3a')
theta_deg = 0.0  # degrees from vertical
e_min = 1.0  # GeV
e_max = 1000000000.0  # GeV
backend = 'auto'

## 02 · Average over $\cos\theta$

Equidistant in $\cos\theta$ from vertical to horizontal, 11 points.

In [ ]:
apply_kernel_and_grid(backend, e_min, e_max, 0)
config.mceq_db_fname = database

angles = np.arccos(np.linspace(1, 0, 11)) * 180.0 / np.pi
mag = 3

mceq_run = MCEqRun(
    interaction_model=interaction_model,
    primary_model=primary_model,
    theta_deg=0.0,
)
e_grid = mceq_run.e_grid

pairs = {
    "mu": ("mu+", "mu-"), "numu": ("numu", "antinumu"),
    "nue": ("nue", "antinue"), "nutau": ("nutau", "antinutau"),
}
prefix = {
    "mu_conv": ("conv", "mu"), "mu_pr": ("pr", "mu"), "mu_total": ("total", "mu"),
    "numu_conv": ("conv", "numu"), "numu_pr": ("pr", "numu"),
    "numu_total": ("total", "numu"),
    "nue_conv": ("conv", "nue"), "nue_pr": ("pr", "nue"),
    "nue_total": ("total", "nue"), "nutau_pr": ("total", "nutau"),
}
flux = {frac: np.zeros_like(e_grid) for frac in prefix}

for theta in angles:
    mceq_run.set_zenith_azimuth(theta)
    mceq_run.solve()
    for frac, (track, group) in prefix.items():
        flux[frac] += sum(
            mceq_run.get_solution(f"{track}_{p}", mag) for p in pairs[group]
        )
flux = {frac: total / len(angles) for frac, total in flux.items()}

## 03 · Spectra

Solid black: total; dash-dot blue: conventional; dashed red: prompt.

In [ ]:
figures = []
for pref, lab in [("numu_", r"\nu_\mu"), ("nue_", r"\nu_e")]:
    fig, ax = plt.subplots(figsize=(4.2, 3))
    ax.loglog(e_grid, flux[pref + "total"], color="k", ls="-", lw=1.5)
    ax.loglog(
        e_grid, flux[pref + "conv"], color="b", ls="-.", lw=1.5,
        label=rf"conventional ${lab}$",
    )
    ax.loglog(
        e_grid, flux[pref + "pr"], color="r", ls="--", lw=1.5,
        label=f"prompt ${lab}$",
    )
    ax.set_xlim(10, 1e7)
    ax.set_ylim(1e-5, 10)
    ax.set_xlabel(rf"$E_{{{lab}}}$ [GeV]")
    ax.set_ylabel(
        r"$\Phi_{" + lab + "}$ (E/GeV)$^{" + str(mag) + " }$"
        + "(cm$^{2}$ s sr GeV)$^{-1}$"
    )
    ax.legend(loc="upper right", frameon=False, numpoints=1)
    fig.tight_layout()
    figures.append(fig)
figures

## 04 · ASCII export

In [ ]:
import os
import tempfile

components = [
    "mu_conv", "mu_pr", "mu_total",
    "numu_conv", "numu_pr", "numu_total",
    "nue_conv", "nue_pr", "nue_total", "nutau_pr",
]
desktop = tempfile.mkdtemp(prefix="mceq_example_")
destination = os.path.join(desktop, "H3a_zenith_av.txt")
np.savetxt(
    destination,
    np.column_stack([e_grid, *[flux[frac] for frac in components]]),
    fmt="%6.5E",
    header=(
        "lepton flux scaled with E**3. Order (E, mu_conv, mu_pr, mu_total, "
        "numu_conv, numu_pr, numu_total, nue_conv, nue_pr, nue_total, "
        "nutau_pr)"
    ),
)
f"Zenith-averaged spectra saved to `{destination}`."